In [1]:
%pip install --upgrade git+https://github.com/ParkhomenkoDV/gte.git@master

  Cloning https://github.com/ParkhomenkoDV/gte.git (to revision master) to /tmp/pip-req-build-_0i_sqjh
  Running command git clone --filter=blob:none --quiet https://github.com/ParkhomenkoDV/gte.git /tmp/pip-req-build-_0i_sqjh
  Resolved https://github.com/ParkhomenkoDV/gte.git to commit 9e889c23176dc1178472973766236dcc47910d2d
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/ParkhomenkoDV/mathematics.git (to revision master) to /tmp/pip-install-h0jcwgv5/mathematics_622d083951fd4b769210c7489bced7b8
  Running command git clone --filter=blob:none --quiet https://github.com/ParkhomenkoDV/mathematics.git /tmp/pip-install-h0jcwgv5/mathematics_622d083951fd4b769210c7489bced7b8
  Resolved https://github.com/ParkhomenkoDV/mathematics.git to commit 60f686efd15b6b21caf492807712a601f4279f00
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/ParkhomenkoDV/thermodynamics.git (to revision master) to /tmp/pip-install-h0jcwgv5/thermodynamics_a22e23aeb598414b90533

# Параметры

In [2]:
from gte import parameters as gtep

'gte/models/compressor_total_temperature.pkl' not found!
'gte/models/compressor_total_pressure.pkl' not found!
'gte/models/compressor_total_pressure_ratio.pkl' not found!
'gte/models/compressor_total_efficiency.pkl' not found!
'gte/models/compressor_power.pkl' not found!
'gte/models/compressor_total_pressure_ratio.pkl' not found!
'gte/models/compressor_total_efficiency.pkl' not found!
'gte/models/compressor_power.pkl' not found!
'gte/models/compressor_outlet_total_temperature_total_pressure_ratio.pkl' not found!
'gte/models/compressor_outlet_total_pressure_total_pressure_ratio.pkl' not found!
'gte/models/compressor_outlet_total_temperature_total_efficiency.pkl' not found!
'gte/models/compressor_outlet_total_pressure_total_efficiency.pkl' not found!
'gte/models/compressor_outlet_total_temperature_power.pkl' not found!
'gte/models/compressor_outlet_total_pressure_power.pkl' not found!
'gte/models/compressor_total_temperature.pkl' not found!
'gte/models/compressor_total_pressure.pkl' not 

In [3]:
for k, v in gtep.items():
  print(f"{k:<15}: {v}")

l              : thermal_conductivity
hc             : heat_capacity
Cp             : heat_capacity_pressure
Cv             : heat_capacity_volume
k              : adiabatic_index
gc             : gas_const
eo             : excess_oxidizing
T              : static_temperature
P              : static_pressure
D              : staticdensity
TT             : total_temperature
PP             : total_pressure
DD             : total_density
m              : mass
v              : volume
a              : sound_speed
a_critical     : critical_sound_speed
c              : absolute_velocity
u              : portable_velocity
w              : relative_velocity
mf             : mass_flow
pipi           : total_pressure_ratio
pi             : static_pressure_ratio
titi           : total_temperature_ratio
ti             : static_temperature_ratio
mach           : mach_number
Nu             : nusselt_number
efficiency     : efficiency
effeff         : total_efficiency
peff           : pressure_efficie

In [47]:
from thermodynamics import atmosphere_standard, gas_const, heat_capacity_p, stoichiometry, lower_heat, gas_const_exhaust_fuel, T0
from substance import Substance

In [5]:
air = Substance(
    "air",
    composition={"N2": 0.78, "O2": 0.21, "Ar": 0.009, "CO2": 0.0004},
    parameters={
        gtep.mf: 50.0,
        gtep.gc: 287.14,
        gtep.TT: 300.0,
        gtep.PP: 101325.0,
        gtep.Cp: 1006.0,
        gtep.k: 1.4,
        gtep.c: 0.0,
    },
    functions={
        gtep.gc: lambda temperature: gas_const("air"),
        gtep.Cp: lambda temperature: heat_capacity_p("air", temperature),
    },
)

In [6]:
from gte import Compressor

In [7]:
compressor = Compressor("АЛ-41Ф")

In [8]:
compressor.variables

{'total_pressure_ratio': nan, 'total_efficiency': nan, 'power': nan}

In [17]:
compressor.total_pressure_ratio = 6

In [10]:
compressor.is_solvable(air)

'getattr(self, gtep.pipi)=6 getattr(self, gtep.effeff)=nan getattr(self, gtep.power)=nan'

In [18]:
compressor.total_efficiency = 0.85

In [19]:
compressor.is_solvable(air)

''

In [20]:
outlet = compressor.solve(air)

In [33]:
for param, value in compressor.summary.items():
  print(f"{param:<30}: {value}")

name                          : АЛ-41Ф
inlet_mass_flow               : 50.0
inlet_gas_const               : 287.14
inlet_total_temperature       : 300.0
inlet_total_pressure          : 101325.0
inlet_heat_capacity_pressure  : 1006.0
inlet_adiabatic_index         : 1.4
inlet_absolute_velocity       : 0.0
outlet_mass_flow              : 50.0
outlet_total_temperature      : 532.2979315327493
outlet_total_pressure         : 607950.0
outlet_gas_const              : 287.14
outlet_heat_capacity_pressure : 1037.2558324484967
outlet_total_density          : 3.9775842583793817
outlet_adiabatic_index        : 1.3827942133453304
outlet_critical_sound_speed   : 421.18679181026033
leak                          : 0
total_pressure_ratio          : 6
total_efficiency              : 0.85
power                         : 11816242.700103628


In [28]:
compressor.validate(0.01)

True

In [29]:
compressor.is_real

True

In [25]:
help(compressor._equations)

Help on method _equations in module gte.compressor:

_equations(x: Tuple[float], args: Dict[str, Any]) -> Tuple method of gte.compressor.Compressor instance
    power - mf * Cp * (T*_outlet - T*_inlet) = 0
    T*_outlet - T*_inlet * (1 + (pi* ** ((k-1) / k)) / eff) = 0
    pi* - T*_outlet / T*_inlet = 0



In [48]:
kerosene = Substance(
    "kerosene",
    composition={"C": 0.85, "H": 0.15},
    parameters={
        gtep.mf: 3,
        gtep.TT: 40 + T0,
        gtep.PP: 101_325,
        "stoichiometry": stoichiometry("kerosene"),
        "lower_heat": lower_heat("kerosene"),
    },
    functions={
        gtep.gc: lambda excess_oxidizing: gas_const_exhaust_fuel(excess_oxidizing, fuel="kerosene"),
        gtep.hc: lambda temperature: 200,
    },
)

In [49]:
from gte import CombustionChamber

In [50]:
cc = CombustionChamber("КС")

In [51]:
cc.variables

{'efficiency_burn': nan, 'pressure_efficiency': nan}

In [52]:
cc.efficiency_burn = 0.99

In [53]:
cc.is_solvable(air, kerosene)

'count_variables=1 + count_outlet_variables=2 > count_equations=2'

In [54]:
cc.pressure_efficiency = 0.95

In [55]:
cc.is_solvable(air, kerosene)

''

In [56]:
outlet = cc.solve(air, kerosene)

In [57]:
for param, value in cc.summary.items():
  print(f"{param:<30}: {value}")

name                          : КС
inlet_mass_flow               : 50.0
inlet_gas_const               : 287.14
inlet_total_temperature       : 300.0
inlet_total_pressure          : 101325.0
inlet_heat_capacity_pressure  : 1006.0
inlet_adiabatic_index         : 1.4
inlet_absolute_velocity       : 0.0
inlet_enthalpy                : 11883.658364440029
outlet_mass_flow              : 53.0
outlet_excess_oxidizing       : 1.1407711613050424
outlet_total_temperature      : 2351.0476377213786
outlet_total_pressure         : 96258.75143436715
outlet_gas_const              : 288.801771908408
outlet_heat_capacity_pressure : 1286.503463000145
outlet_total_density          : 0.14176823800305463
outlet_adiabatic_index        : 1.289467056623294
outlet_critical_sound_speed   : 874.5478275537812
leak                          : 0
efficiency_burn               : 0.99
pressure_efficiency           : 0.95
fuel_mass_flow                : 3
fuel_total_temperature        : 313.15
fuel_total_pressure        

In [58]:
cc.validate(0.01)

True

In [59]:
cc.is_real

True

In [68]:
data = []
for pi in range(6, 12, 1):
  for eff in range(85, 90, 1):
    c = Compressor()
    c.total_pressure_ratio = pi
    c.total_efficiency = eff/100
    outlet = c.solve(air)
    if c.is_real:
      data.append(c.summary)
len(data)

30

In [69]:
%pip install pandas

In [70]:
import pandas as pd

In [71]:
pd.DataFrame(data).T

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
name,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,...,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor,Compressor
inlet_mass_flow,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,...,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
inlet_gas_const,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14,...,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14,287.14
inlet_total_temperature,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0,...,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0,300.0
inlet_total_pressure,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,...,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0,101325.0
inlet_heat_capacity_pressure,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,...,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0,1006.0
inlet_adiabatic_index,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,...,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4,1.4
inlet_absolute_velocity,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
outlet_mass_flow,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,...,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0,50.0
outlet_total_temperature,532.297932,529.661661,527.084298,524.563893,522.098586,557.593375,554.682402,551.836179,549.052573,546.329546,...,619.640368,616.068073,612.574353,609.156645,605.812498,637.06524,633.310403,629.637902,626.045058,622.529309
